In [1]:
"""
import necessary modules
"""
from scipy.interpolate import griddata, RBFInterpolator
import numpy as np
from dorna2 import Dorna
from camera import Camera
from dorna_vision import Detection
import time
from dorna_vision import grasp
from dorna2 import pose as dorna_pose

"""
parameters
"""
robot_ip_address = "10.0.0.100" # robot ip address
output_config = {
    "pick": [0, 1, 0],
    "pick_init": [0, 1, 0],
    "close": [[0, 1]],
    "open": [[0, 0]],
}

tool = {
    "vial": [0, 0, 55, 0, 0, 45], #3
}

frame = {
    "place": [320, -197, 40, -180, 0, 0]
}

imaging_joint = [0, 55.656738, -143.393555, 0, -2.263184, 0]
middle_joint = imaging_joint

# detection parameters
detection_prm = {
    "microplate":{
            'roi': {'corners': [[170, 1], [170, 400], [710, 400], [710, 1]], 'inv': False, 'crop': False}, 
            'detection': {'cmd': 'kp', 'path': 'model/microplate_keypoint.pkl', 'conf': 0.5, 'cls': {}},
            'sort': {'cmd': 'shuffle', 'max_det': 1},
            'display': {'label': 0, 'save_img': False, 'save_img_roi': False},
            },
    "vial":{
            'detection': {'cmd': 'od', 'path': 'model/vial.pkl', 'conf': 0.5, 'cls': []}, 
            'sort': {'cmd': 'shuffle', 'max_det': 1},
            'display': {'label': 0, 'save_img': False, 'save_img_roi': False},
            }
    }



# good candidate
xyz_min = 33
aspect_ratio = 0.9

# grasp
rvec_base = [180, 0, 0]
finger_location = [0+45, 90+45, 180+45, 270+45]
grasp_padding = 0 # pixel 3 was 0
gripper_opening = 14 # mm 12 was 16
finger_width = 1 # mm
gripper_thickness = 8 # 8 pxl was 9
bb_radius = 25 # pxl
pick_samples = 128
search_rotation = [-45, 50]
gripper_rotation = [
    {"axis": [0, 0, 1], "angle": 0},
    {"axis": [0, 0, 1], "angle": 90},
    {"axis": [0, 0, 1], "angle": 180},
    {"axis": [0, 0, 1], "angle": 270},
]
rvec_error_thr = 20 # mm
xyz_error_thr = 5 # mm

# new data
pxl_ref_new = []
xyz_ref_new = []

############ pick n place
config_pick_n_place= {
    "pick": {
        "type": "world", # robot, world, joint
        "loc": [None, [0, 0]],
        "output": output_config["close"],
        "cmd":2*[{"cmd":"output", "out"+str(output_config["pick"][0]): output_config["pick"][2], "queue":0}, 
                    {"cmd": "sleep", "time": 0.2}, 
                    {"cmd":"output", "out"+str(output_config["pick"][0]): output_config["pick"][1], "queue":0}, 
                    {"cmd": "sleep", "time": 0.2}, 
                    {"cmd":"output", "out"+str(output_config["pick"][0]): output_config["pick"][2], "queue":0}, 
                    {"cmd": "sleep", "time": 0.2}
        ],
        "approach": [[0, 0, -10, 0, 0, 0]],
        "exit": [[0, 0, -60, 0, 0, 0]],
    },
    "end": {
        "type": "joint", # robot, world, joint
        "loc": [imaging_joint, [0, 0]],
        },
    "sleep": 0.5, 
    "motion": "lmove", "speed": 0.1, "cont": 1, "corner": 250, 
    "freedom": {"num":20, "range":[0.01, 0.01, 0.01], "early_exit":False }, "timeout": -1, "sim":0,
}

In [2]:
pxl_ref = np.array([[238, 37], [659, 51], [653, 366], [229, 356], [620, 70], [571, 76], [524, 76], [518, 122], [569, 123], [677, 76], [674, 129], [680, 180], [677, 234], [682, 287], [681, 328], [623, 363], [625, 332], [628, 279], [626, 227], [629, 172], [577, 38], [572, 122], [577, 176], [573, 231], [575, 274], [572, 326], [569, 353], [518, 347], [520, 325], [521, 275], [522, 226], [518, 173], [521, 117], [520, 42], [474, 30], [466, 73], [466, 124], [467, 181], [468, 229], [471, 280], [467, 335], [471, 362], [411, 361], [414, 330], [413, 282], [412, 229], [415, 178], [416, 122], [416, 73], [415, 36], [366, 36], [363, 72], [365, 122], [362, 178], [364, 228], [365, 280], [360, 331], [361, 353], [308, 350], [309, 325], [315, 285], [310, 228], [312, 179], [309, 118], [316, 67], [311, 43], [263, 44], [263, 127], [264, 225], [254, 278], [256, 330], [240, 355], [222, 320], [218, 278], [228, 222], [235, 168], [244, 131], [223, 64]])
xyz_ref = np.array([[347.7095368787692, 102.96666293756756, 34.99999999999999], [340.90584449846625, -89.33890192406768, 35.00000000000001], [196.13442761829998, -86.60786886781126, 35.000000000000014], [200.87083908898688, 106.7856316195485, 35.0], [332.35046734844894, -71.559260569004, 34.99999999999999], [329.55555120118356, -49.01777925228525, 35.00000000000001], [329.41456935762176, -28.25009719852554, 34.999999999999986], [308.05558849576653, -25.02997936327286, 35.00000000000001], [307.3307488672136, -48.02761110036437, 34.999999999999986], [329.435920531179, -97.8452103636779, 35.00000000000001], [306.03452794913113, -96.69564992623292, 34.99999999999999], [281.77911251174123, -98.88623058931218, 34.99999999999999], [257.2299674262005, -98.19320408777057, 34.99999999999999], [233.7107050942172, -100.41799834017081, 35.0], [213.7803151205916, -99.31834503380927, 35.00000000000001], [197.8673208181397, -73.03875466147981, 35.00000000000003], [211.37083865407453, -74.45477807521073, 35.0], [236.4407608074056, -75.29249186258187, 34.999999999999986], [260.7137322300195, -73.76272548116808, 35.0], [284.8800530319003, -74.99772556479012, 35.0], [346.54801901620715, -51.76741951439114, 35.00000000000003], [308.24784714499464, -49.64732587683103, 34.999999999999986], [282.96580242560054, -51.379005690153235, 34.99999999999998], [258.6319937688357, -49.123744732968994, 35.0], [238.63527262856843, -50.47957630299139, 34.99999999999997], [214.7285245531388, -48.97036168197315, 35.0], [201.64326610516827, -47.57105415547389, 35.00000000000002], [204.6738591036463, -24.240319533448243, 35.0], [215.1742463142637, -24.878878817110568, 34.999999999999986], [237.84550563722019, -25.876231386788493, 35.000000000000014], [261.0453522268943, -26.478702048124703, 35.00000000000003], [284.26207166698737, -24.888391597502906, 34.999999999999986], [310.3235981058526, -26.34426157634131, 35.000000000000014], [344.60459754391456, -26.36328632234038, 35.000000000000014], [349.84663805630663, -5.726426055372246, 35.00000000000001], [331.07910181513853, -2.269158578772256, 35.0], [307.2957901837243, -2.0243482858227266, 35.0], [280.58231481856507, -2.3650684103825506, 35.00000000000001], [259.4771425111734, -2.5247022834552038, 35.0], [235.92879399656016, -3.4854870556394753, 34.99999999999998], [210.5214310073676, -1.6639331100894976, 35.00000000000001], [198.68901538544267, -3.105030687420579, 35.000000000000014], [198.67149061462968, 23.825008167590525, 35.0], [212.9015180061884, 22.22847724674203, 35.0], [235.38289345166964, 21.973697510448, 35.00000000000001], [259.849910628346, 22.988427042010983, 35.0], [282.2751320972648, 21.788595773613878, 35.000000000000014], [307.9220912477675, 21.18438205991484, 34.99999999999999], [330.98246209326857, 21.159025480548394, 34.99999999999999], [347.5894667166354, 21.68785404955383, 35.000000000000014], [347.4574806969967, 44.62179235345162, 35.00000000000001], [331.8227950831096, 46.05145911958485, 35.00000000000001], [308.5018707134087, 45.298176368665615, 35.000000000000014], [282.4847736710587, 46.435047556980514, 35.00000000000001], [260.47255254415, 45.5917268294784, 34.99999999999999], [236.1133768055617, 45.07826335474502, 35.0], [212.23285723103646, 47.566237565775445, 34.99999999999999], [201.58370825012332, 47.166081394228854, 34.999999999999986], [203.7624335705029, 70.85904146366387, 35.0], [215.4342322247358, 70.7091383745878, 34.99999999999999], [234.38035804732598, 68.02285655283072, 35.00000000000001], [260.1439930364939, 70.13623556613841, 35.0], [281.5988029833642, 69.31565687808153, 34.99999999999999], [310.62236979566745, 70.43730724110493, 35.00000000000001], [333.40042725491736, 67.86271228705687, 35.00000000000001], [344.63125986040893, 69.96687551741555, 35.00000000000002], [344.55513051591697, 91.83967946700268, 35.00000000000001], [306.8738453599848, 91.56845103584415, 34.99999999999999], [261.45800019465156, 91.11704746085819, 35.0], [237.40767401109167, 94.96657425831265, 34.99999999999999], [213.02581296264566, 94.6645665949559, 35.00000000000001], [201.19739866724794, 102.38393388803574, 35.0], [217.23374595507693, 111.52149858809781, 34.999999999999986], [238.17673160104397, 111.86342287568353, 35.0], [262.4716399312058, 107.16210318760133, 34.999999999999986], [287.5002033155357, 102.95773250658105, 34.999999999999986], [305.48937202403044, 100.00866924436285, 34.999999999999986], [336.2615179918474, 109.71248896076193, 34.99999999999998]])

In [3]:
"""
initialize the robot, camera, and object detection
"""
robot = Dorna() # initialize robot
robot.connect(robot_ip_address) # connect to robot

camera = Camera() # initialize camera
camera.connect() # connect to camera

detection_microplate = Detection(camera=camera, robot=robot, **detection_prm["microplate"]) # initialize the object detection
detection_vial = Detection(camera=camera, robot=robot, **detection_prm["vial"]) # initialize the object detection

c:\Users\hossein\AppData\Local\Programs\Python\Python39\lib\importlib\util.py:245: DeprecationWarning: The `openvino.runtime` module is deprecated and will be removed in the 2026.0 release. Please replace `openvino.runtime` with `openvino`.
  self.__spec__.loader.exec_module(self)


In [9]:
"""
init robot
"""
robot.tool(tool["vial"]) # set the tcp
robot.set_output(output_config["pick_init"][0], output_config["pick_init"][2]) # set output
robot.set_motor(1) # turn on the robot motors
robot.sleep(1) # sleep to allow the robot to settle before moving

#set safe initial position
robot.go(joint=imaging_joint, speed=config_pick_n_place["speed"], sim=0)

"""
run the object microplate detection and pick and place
"""
rounds = 1
for j in range(rounds):
    print("round ", j, ": move plate to a new position")
    time.sleep(1)
    for i in range(2):
        # always run this to make sure that the rbot is stationary before running the object detection
        time.sleep(0.1)
        result_microplate = detection_microplate.run()
        
        # init pose estimator
        bbox_microplate = [r["corners"] for r in result_microplate if r["cls"] == "microplate"]
        if not len(bbox_microplate):
            print("no microplate")
            continue
        bbox_microplate = bbox_microplate[0]
        
        # vial detection
        result_vial = detection_vial.run(roi={'corners': bbox_microplate, 'inv': False, 'crop': True, "offset": rvec_error_thr})
        if not result_vial:
            print("no vial")
            continue

        # assign tvec and xyz
        index = 0

        # pick
        if i == 0:
            # First try cubic griddata
            xyz_interp = griddata(pxl_ref, xyz_ref, result_vial[index]["center"], method="cubic")

            # If NaN, fall back to RBFInterpolator
            if xyz_interp is None or np.any(np.isnan(xyz_interp)):
                print("outside grid")
                rbf = RBFInterpolator(pxl_ref, xyz_ref, kernel="thin_plate_spline")
                xyz_interp = rbf([result_vial[index]["center"]])

            # tvec and xyz
            result_vial[index]["tvec"] = [xyz_interp[0][0], xyz_interp[0][1], xyz_interp[0][2]]
            result_vial[index]["xyz"] = [xyz_interp[0][0], xyz_interp[0][1], xyz_interp[0][2]]
            
            # best_pick
            best_rvec_pick = grasp.collision_free_rvec(
                result_vial[index]["id"], 
                rvec_base, 
                gripper_opening,
                finger_width,
                finger_location, 
                detection_vial, mask_type="elp", prune_factor=4, num_steps=360)        
            if best_rvec_pick is None:
                print("no grasp found")
                continue
            print("rvec: ", best_rvec_pick)
            # all the possible solutions
            pose_all = [result_vial[index]["tvec"] + dorna_pose.rotate_abc(best_rvec_pick, axis=rotation["axis"], angle=rotation["angle"], local=True) for rotation in gripper_rotation]
            
            # nearest pose
            best_pose = robot.kinematic.nearest_pose(pose_all, 
                                                    detection_vial.retval["camera_data"]["joint"], 
                                                    config_pick_n_place["freedom"])
            if best_pose is None:
                print("no nearest pose found")
                continue

            # pick pose
            config_pick_n_place["pick"]["loc"][0] = best_pose
            config_pick_n_place["pick"]["tool"] = 2 * [tool[result_vial[index]["cls"]], tool[result_vial[index]["cls"]]]

            retval = robot.pick_n_place(**config_pick_n_place)

            # append ref data
            new_xyz = np.array(best_pose[0:3]).tolist()
        elif i == 1: 
            pxl_ref_new.append(result_vial[index]["center"])
            xyz_ref_new.append(new_xyz)
            print("new point added")

round  0 : move plate to a new position
rvec:  [179.99999879258172, -1.3411044984503219e-06, -2.322861129575134e-06]
new point added


In [127]:
print(xyz_ref_new)
print(pxl_ref_new)
len(xyz_ref_new) == len(pxl_ref_new)
print(len(xyz_ref_new))

[[359.70523818030273, -98.80338561913848, 35.00000000000002], [340.76478571375054, -99.02001311025965, 35.00000000000001], [322.83925353197964, -100.27964924945445, 34.99999999999999], [306.0136282292527, -100.83938665377464, 34.99999999999999], [296.41717883743974, -83.246977330635, 35.00000000000001], [313.4423856681421, -83.67866149723541, 34.99999999999998], [331.3715939912409, -83.97050259565154, 34.999999999999986], [349.4394622990883, -83.71931918377956, 35.000000000000014], [357.35677580263206, -65.37981255292952, 35.00000000000003], [339.01983186278414, -66.37156547324707, 35.000000000000014], [321.6589525365677, -67.43823153274138, 34.999999999999986], [302.8153328081121, -69.41140882339572, 34.99999999999999], [292.7992642506922, -52.62224888349954, 34.99999999999997], [311.0026808525967, -53.33203307722376, 34.999999999999986], [328.77398671013975, -54.074336667022145, 35.0], [346.12466735747995, -54.47761888714723, 35.00000000000001], [353.9722550562511, -36.07990580531353

In [111]:
pxl_ref_new.pop()
xyz_ref_new.pop()

[192.1480301624751, -63.09230641731264, 35.000000000000036]

In [5]:
robot.close() # close robot
camera.close() # close camera
detection_microplate.close() # close object detection
detection_vial.close()